In [19]:
import pandas as pd
import re
import hashlib

In [20]:
chat = "private"

In [21]:
def hash_author(author: str) -> str:
    """Return a short anonymized identifier for an author."""
    # Use SHA1 hash and take first 8 characters
    h = hashlib.sha1(author.encode('utf-8')).hexdigest()[:8]
    return str(h)

In [22]:
df = pd.read_csv(f"../data/raw/{chat}.csv")

# anonimize
df['Author'] = df['Author'].apply(hash_author)

# remove attachments
df['Content'] = df.apply(
    lambda row: f"<ATTACH> {row['Content'] if pd.notna(row['Content']) else ''}".strip()
    if pd.notna(row['Attachments']) and row['Attachments'].strip() != '' 
    else row['Content'],
    axis=1
)

# remove links
url_regex = r"(?P<url>https?://[^\s]+)"
df['Content'] = df['Content'].apply(
	lambda x: re.sub(url_regex, "<URL>", x)
	if isinstance(x, str) else x,
)

date_mask = df['Date'].between('2021-01-01', '2023-12-31')
df = df.loc[date_mask]

df = df[['Author', 'Content']]
df['Content'] = df['Content'].astype("string")
print(df.head())

         Author                Content
62025  fa09cb53         new year bitch
62026  fa09cb53               eat piss
62027  fa09cb53                and cum
62028  fa09cb53  wholesome 2021 moment
62029  e1688f3c       the year is 2021


In [23]:


df.to_csv(f'../data/processed/{chat}.csv', index=False)